# **PyTorch Day-6**
**In this notebook we will learn how to implement a transformer based GPT model**

**Our goal is to build a model with gpt2 structure**

In [2]:
try:
    import datasets
except ImportError:
    !pip install datasets
    import datasets

In [3]:
try:
    import transformers
except ImportError:
    %pip install transformers
    import transformers

In [44]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

* # Checking for gpu
**Using Apple Silicon GPU (MPS)**

In [5]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU.")

Using Apple Silicon GPU (MPS)


* # Data
**We use the TinyStories data set**

In [6]:
train_data = datasets.load_dataset("roneneldan/TinyStories", split="train")
test_data = datasets.load_dataset("roneneldan/TinyStories", split="validation")
print('------------------------------------------------------------')
print('Training data:\n',train_data)
print('Testing data:\n',test_data)
print('------------------------------------------------------------')
print('An example of training data:\n',train_data[0])
print('An example of testing data:\n',test_data[0])

------------------------------------------------------------
Training data:
 Dataset({
    features: ['text'],
    num_rows: 2119719
})
Testing data:
 Dataset({
    features: ['text'],
    num_rows: 21990
})
------------------------------------------------------------
An example of training data:
 {'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and w

* # Tokenization
**For tokenization we use the gpt2 tokenizer**

In [7]:
from transformers import AutoTokenizer
# Load the GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT-2 does not have a padding token.
# We use the End of sentence (EOS) token for padding
tokenizer.pad_token = tokenizer.eos_token

# Defining a tokenizing function
def tokenizing_fn(data):
    return tokenizer(
        data["text"],
        truncation=True,
        max_length=512,
        padding= False
    )

# Tokenize the datasets
train_data_tokens = train_data.map(
    tokenizing_fn,
    batched=True,
    remove_columns=["text"],
    batch_size= 32
)

test_data_tokens = test_data.map(
    tokenizing_fn,
    batched=True,
    remove_columns=["text"],
    batch_size= 32
)

In [8]:
total_train_tokens = sum(
    len(story["input_ids"])
    for story in train_data_tokens
)


total_test_tokens = sum(
    len(story["input_ids"])
    for story in test_data_tokens
)

print("Total training tokens:", total_train_tokens)

print("Total testing tokens:", total_test_tokens)

Total training tokens: 463939980
Total testing tokens: 4675638


In [9]:
# Looking at an example of data
story = test_data_tokens[2000]
print(story.keys())
token_ids = story["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(token_ids)

for token_id, token in zip(token_ids, tokens):
    print(f"story token ids = {token_id:6d} , story tokes = {token}")

dict_keys(['input_ids', 'attention_mask'])
story token ids =  14295 , story tokes = Jack
story token ids =    373 , story tokes = Ġwas
story token ids =    379 , story tokes = Ġat
story token ids =    262 , story tokes = Ġthe
story token ids =  10481 , story tokes = Ġbeach
story token ids =    351 , story tokes = Ġwith
story token ids =    465 , story tokes = Ġhis
story token ids =   1641 , story tokes = Ġfamily
story token ids =     13 , story tokes = .
story token ids =    679 , story tokes = ĠHe
story token ids =  12408 , story tokes = Ġwore
story token ids =    465 , story tokes = Ġhis
story token ids =    649 , story tokes = Ġnew
story token ids =   4171 , story tokes = Ġblue
story token ids =  12581 , story tokes = Ġpants
story token ids =    475 , story tokes = Ġbut
story token ids =    339 , story tokes = Ġhe
story token ids =   2936 , story tokes = Ġfelt
story token ids =    845 , story tokes = Ġvery
story token ids =  12916 , story tokes = Ġuncomfortable
story token ids =    

* ### **Preparing the data to pass to a torch model**

*  **Let's concatenate the data in torch tensors**

In [15]:
batch_size = 32
context_size = 64

# total_train_tokens = total_train_tokens // 10
# total_test_tokens = total_test_tokens // 10


train_set = torch.zeros(total_train_tokens, dtype=torch.int64)
test_set = torch.zeros(total_test_tokens, dtype=torch.int64)

start_index = 0
end_index = 0
for i , story in enumerate(train_data_tokens):
    story_tensor = torch.tensor(story["input_ids"]).to(torch.int64)
    end_index = end_index + story_tensor.shape[0]  
    train_set[start_index:end_index] = story_tensor[:]
    start_index = end_index

start_index = 0
end_index = 0
for i , story in enumerate(test_data_tokens):
    story_tensor = torch.tensor(story["input_ids"])
    end_index = end_index + story_tensor.shape[0]  
    test_set[start_index:end_index] = story_tensor[:]
    start_index = end_index


n_test = (test_set.shape[0]//(batch_size * context_size)) * (batch_size * context_size)
n_train = (train_set.shape[0]//(batch_size * context_size)) * (batch_size * context_size)

train_set = train_set[:n_train]
test_set = test_set[:n_test]

* **Reshaping the training data into the format to pass to the embedding layer in the batch loop**
* It should be a tensor of shape (N_batches, batch_size =32, context_size=64) 

In [52]:
N_batches = n_train // (batch_size * context_size)
print(f"Number of batches={N_batches}")
train_set_r = train_set.reshape(N_batches , batch_size , context_size)
train_set_r.shape
vocab_size = train_set_r.max()
print(f"Vocab size= {vocab_size}")
print(f"Shape of the training set={train_set_r.shape}")

Number of batches=226533
Vocab size= 50255
Shape of the training set=torch.Size([226533, 32, 64])


* ### Sending the data to device

In [210]:
train_set_r = train_set_r.to(device)

* # Creating the model

* ### Model configuration

In [298]:
@dataclass
class DataConfig:
    N_batches: int 
    batch_size: int
    vocab_size: int

@dataclass
class ModelConfig:
    context_size: int
    embed_dim: int
    init_std: float
    eps: float
    N_heads: int
    attn_dim: int
    dropout: float
    
    
d_cfg = DataConfig(N_batches =  N_batches,
                   batch_size = batch_size,
                   vocab_size = vocab_size.item(),
                  ) 

m_cfg = ModelConfig(context_size =context_size,
                    embed_dim=512,
                    init_std=0.02,
                    eps=1e-5,
                    N_heads = 4,
                    attn_dim = 128,
                    dropout = 0.1
                   )

print(m_cfg)
print(d_cfg)

ModelConfig(context_size=64, embed_dim=512, init_std=0.02, eps=1e-05, N_heads=4, attn_dim=128, dropout=0.1)
DataConfig(N_batches=226533, batch_size=32, vocab_size=50255)


* ### The embedding layer

In [268]:
class Embedding(nn.Module):
    def __init__(self, m_cfg:ModelConfig , d_cfg:DataConfig):
        super().__init__()
        vocab_size = d_cfg.vocab_size 
        embed_dim = m_cfg.embed_dim
        context_size = m_cfg.context_size

        # Token embedding
        self.token_embedding = nn.Parameter(
            torch.empty(vocab_size, embed_dim)
        )
        # Positional embedding
        self.position_embedding = nn.Parameter(
            torch.empty(context_size, embed_dim)
        )

        # Initialization
        nn.init.normal_(self.token_embedding, mean=0.0, std=m_cfg.init_std)
        nn.init.normal_(self.position_embedding, mean=0.0, std=m_cfg.init_std)

    def forward(self, input_ids):

        position_ids = torch.arange(context_size, device=input_ids.device)
        
        x = self.token_embedding[input_ids] + self.position_embedding[position_ids]

        return x       

**Taking a look at the embedding layer**

In [269]:
model_embed = Embedding(m_cfg , d_cfg)
model_embed.to(device)
model_embed.eval()
X=train_set_r[0,:,:].squeeze()
print(f"Input is on device:{X.device}")
model_embed.eval()
with torch.inference_mode():
    Y_embed = model_embed(X)
print(f"Shape of the embedding layer output= {Y_embed.shape}")
model_embed.state_dict()
print(f"Output is on device:{Y_embed.device}")

Input is on device:mps:0
Shape of the embedding layer output= torch.Size([32, 64, 512])
Output is on device:mps:0


* # The normalization layer

In [270]:
class Normalize(nn.Module):
    def __init__(self, m_cfg:ModelConfig):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(m_cfg.embed_dim))
        self.beta = nn.Parameter(torch.zeros(m_cfg.embed_dim))
        self.eps = m_cfg.eps
        
    def forward(self, x: torch.Tensor):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        x = (x - mean) / torch.sqrt(var + self.eps)

        return self.gamma * x + self.beta

**Looking at the Normalization layer**

In [271]:
model_norm = Normalize(m_cfg)
model_norm.to(device)
model_norm.eval()
with torch.inference_mode():
    Y_norm = model_norm(Y_embed)

print(f"Normalization layer output shape is {Y_norm.shape}")
print(f"Output is on device:{Y_norm.device}")
model_norm.state_dict()


Normalization layer output shape is torch.Size([32, 64, 512])
Output is on device:mps:0


OrderedDict([('gamma',
              tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1.,

* # The Attention layer
  **This is the most exciting part of the transformer**

In [301]:
class Attention(nn.Module):
    def __init__(self, m_cfg:ModelConfig):
        super().__init__()
        
        
        self.W_keyes = nn.Parameter(
            torch.empty((m_cfg.N_heads, m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map the embedded model vectors to attention keyes

        self.b_keyes = nn.Parameter(
            torch.zeros((m_cfg.N_heads, m_cfg.attn_dim))
        ) #<- bias for keyes

        self.W_queries = nn.Parameter(
            torch.empty((m_cfg.N_heads, m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map the embedded model vectors to attention queries

        self.b_queries = nn.Parameter(
            torch.zeros((m_cfg.N_heads, m_cfg.attn_dim))
        ) #<- bias for queries
        
        self.W_values = nn.Parameter(
            torch.empty((m_cfg.N_heads, m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map the embedded model vectors to attention values

        self.b_values = nn.Parameter(
            torch.zeros((m_cfg.N_heads, m_cfg.attn_dim))
        ) #<- bias for values
        
        self.W_output = nn.Parameter(
            torch.empty((m_cfg.N_heads * m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map from attention space back to embedding space 
        
        self.b_output = nn.Parameter(
            torch.zeros((m_cfg.embed_dim))
        ) #<- output bias

        # initializing with standard Gaussian random numbers
        nn.init.normal_(self.W_queries, std=m_cfg.init_std)
        nn.init.normal_(self.W_keyes, std=m_cfg.init_std)
        nn.init.normal_(self.W_values, std=m_cfg.init_std)
        nn.init.normal_(self.W_output, std=m_cfg.init_std)

        
        self.register_buffer("neg_inf",
                             torch.tensor(float("-inf"),
                             dtype=torch.float32)
                            ) #<- will be used later for causal masking
        
        self.register_buffer("mask",
                            torch.triu(
                                torch.ones(size=(m_cfg.context_size , m_cfg.context_size) , device=device ),
                                diagonal=1).bool()
                            ) #<- The causal mask
        
        self.m_cfg = m_cfg #<- owning the cfg info
        self.attention_dropout = nn.Dropout(m_cfg.dropout)
        
    def forward(self, x):

        # Mapping vectors from the embedding space to attention space.
        ## for the following we can also use torch.einsum() 
        x_k = torch.tensordot(x, self.W_keyes, dims=([-1], [-1])) + self.b_keyes   
        x_q = torch.tensordot(x, self.W_queries,  dims=([-1], [-1])) + self.b_queries
        x_v = torch.tensordot(x, self.W_values, dims=([-1], [-1])) + self.b_values
        # print(f"Shape of x_v is {x_v.shape}")

        
        attn_scores = torch.einsum(
            'bqha,bkha -> bhqk',
            x_q , x_k) #<- contract on the attention dimension (dot product of queries and keyes)
        
        attn_scores /= self.m_cfg.attn_dim**0.5 #<- Normalizing with square root of attention space dimension

        q = x.shape[1]
        attn_scores = attn_scores.masked_fill_(
            self.mask[:q,:q],
            self.neg_inf
        ) #<- applying the causal mask
        # print(f"Shape of attention scores is {attn_scores.shape}")
        
        attn_probs = attn_scores.softmax(dim = -1) #<- scores to probabilities so that sum of each row adds up to one
        attn_probs = self.attention_dropout(attn_probs)

        V = torch.einsum(
            'bhqv,bvha->bqha',
            attn_probs , x_v
        )#<- computing the weighted values 
        
        b, q, h, a = V.shape
        V = V.reshape(b, q, h * a) #<- Concatenating all the heads
        # print(f"Shape of wighted values is {V.shape}")
        
        output = torch.einsum(
            'ce,bvc->bve',
            self.W_output , V
        ) + self.b_output
        return output

In [302]:
# Looking at the Attention class
model_attn = Attention(m_cfg)
model_attn.to(device)
model_attn.eval()
print(f"Shape of the Keyes matrix = {model_attn.W_keyes.shape}")
print(f"Shape of the input matrix = {Y_norm.shape}")
with torch.inference_mode():
    output = model_attn(Y_norm[:,:,:])
print(f"Attention layer output's shape is {output.shape}")


Shape of the Keyes matrix = torch.Size([4, 128, 512])
Shape of the input matrix = torch.Size([32, 64, 512])
Attention layer output's shape is torch.Size([32, 64, 512])
